# 02 — Evaluate All FlanT5 Conditions

Evaluates all FlanT5 conditions on the PlanBench test set and reports
plan-level F1-macro, action-level F1-macro, and failing step accuracy.

## Conditions evaluated

**Direct format** (full plan → single valid/invalid verdict):
- `FlanT5_Direct` — zero-shot base model, direct format
- `FlanT5_Direct_FT50` — fine-tuned on 50 fewshot plans, direct format
- `FlanT5_Direct_FT_Random` — fine-tuned on random 70% of test set (own 20% holdout)

**Stepwise format** (action by action → pipeline verdict):
- `FlanT5_Stepwise` — zero-shot base model, stepwise format
- `FlanT5_Stepwise_FT_FOLIO` — fine-tuned on FOLIO only
- `FlanT5_Stepwise_FT_PB` — fine-tuned on PlanBench fewshot only
- `FlanT5_Stepwise_FT_Combined` — proposed method (FOLIO + PlanBench)

## Metrics

- **Plan F1-macro** — plan-level binary classification (valid/invalid), macro-averaged
- **Action F1-macro** — step-level binary classification (A/B), stepwise conditions only
- **Failing Step Accuracy** — fraction of invalid plans where the model correctly
  identifies the first failing action step, stepwise conditions only

## Output

- `paper2/results/results_flant5.json` — metrics for all conditions
- `paper2/results/raw_preds_flant5.json` — raw prediction arrays for all conditions
  (for computing additional metrics without re-running inference)

## Prerequisites

Run notebooks 00, 01a, 01b, 01c first.

## 1. Setup

DEBUG mode evaluates only `DEBUG_N` plans per domain — useful for verifying
the pipeline runs end-to-end before committing to the full evaluation (~2-3 hours).
Set `DEBUG=False` for the full run.

In [ ]:
DEBUG   = False   # Set False for full evaluation
DEBUG_N = 10      # plans per domain in debug mode
print(f'Mode: {"DEBUG" if DEBUG else "FULL EVAL"}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U transformers peft accelerate scikit-learn bitsandbytes

In [ ]:
import os, gc, json, re, copy, random, warnings
import numpy as np
import torch
from collections import defaultdict, Counter
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)

# -- Paths -------------------------------------------------------------------
ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
DATA_DIR    = f'{ROOT}/data'
RESULTS_DIR = f'{ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

SW_TEST_PATH     = os.path.join(DATA_DIR, 'planbench_stepwise_test.jsonl')
DI_TEST_PATH     = os.path.join(DATA_DIR, 'planbench_direct_test.jsonl')
SPLITS_PATH      = os.path.join(DATA_DIR, 'direct_splits.json')

# Adapter paths
ADAPTER_COMBINED = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_Combined', 'lora_adapter')
ADAPTER_FOLIO    = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_FOLIO',    'lora_adapter')
ADAPTER_PB       = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_PB',       'lora_adapter')
ADAPTER_FT50     = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT50',           'lora_adapter')
ADAPTER_RANDOM   = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Random',      'lora_adapter')
ADAPTER_LENGTH   = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Length',      'lora_adapter')

# Tokenizer paths (saved alongside adapters)
TOK_COMBINED = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_Combined', 'tokenizer')
TOK_FOLIO    = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_FOLIO',    'tokenizer')
TOK_PB       = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_PB',       'tokenizer')
TOK_FT50     = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT50',           'tokenizer')
TOK_RANDOM   = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Random',      'tokenizer')
TOK_LENGTH   = os.path.join(ROOT, 'adapters', 'FlanT5_Direct_FT_Length',      'tokenizer')

# -- Model config ------------------------------------------------------------
BASE_MODEL     = 'google/flan-t5-xl'
MAX_SOURCE_LEN_SW = 1024   # stepwise: longest prompt ~600 tokens
MAX_SOURCE_LEN_DI = 2048   # direct: logistics needs ~1800 tokens
MAX_TARGET_LEN = 4
BATCH_SIZE     = 32     # inference only - no gradient storage needed

DOMAINS = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']

use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
def cleanup(): gc.collect(); torch.cuda.empty_cache()

print(f'ROOT      : {ROOT}')
print(f'GPU       : {torch.cuda.get_device_name(0)}')
print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'BF16      : {use_bf16}')
print(f'BATCH_SIZE: {BATCH_SIZE}')

## 2. State Helpers and Goal Check

The stepwise evaluator tracks the world state as actions are verified.
After each action predicted valid (A), the state is updated deterministically.
When all actions pass, a final goal check determines whether the plan is valid.

These helpers are copied verbatim from `00_data_preparation.ipynb` to ensure
consistent state tracking and Mystery vocabulary abstraction across all notebooks.

- Mystery abstraction: same `rel1..rel5` mapping as notebook 00
- Blocksworld state: `_bw_parse`, `_bw_apply`
- Mystery state: `_mystery_apply`
- Logistics state: `_logistics_parse`, `_logistics_apply`
- `goal_check(plan_recs, domain, goal_facts)` — called when pipeline predicts
  all A; returns 1 (valid) or 0 (goal not satisfied)

In [ ]:
import re as _re

# -- Mystery abstraction (rel1..rel5) ---------------------------------------
# Must match 00_data_preparation.ipynb exactly.
_MYSTERY_FULL_MAP = {
    'attack':'action1', 'succumb':'action2',
    'overcome':'action3', 'feast':'action4',
    'Attack':'Action1', 'Succumb':'Action2',
    'Overcome':'Action3', 'Feast':'Action4',
    'Province':'Rel1', 'province':'rel1',
    'Planet':'Rel2',   'planet':'rel2',
    'Harmony':'Rel3',  'harmony':'rel3',
    'Pain':'Rel4',     'pain':'rel4',
    'Craves':'Rel5',   'craves':'rel5',
}

def _abstract_mystery(text):
    for orig, repl in _MYSTERY_FULL_MAP.items():
        text = _re.sub(r'\b' + _re.escape(orig) + r'\b', repl, text)
    return text

# -- Blocksworld state helpers ------------------------------------------------
# State is a dict: {on, on_table, clear, holding, hand_empty}
def _bw_parse(facts):
    s = {'on':[], 'on_table':[], 'clear':[], 'holding':None, 'hand_empty':True}
    for f in facts:
        fl = f.lower().rstrip('.')
        m = _re.search(r'the (\w+) block is on top of the (\w+) block', fl)
        if m: s['on'].append((m.group(1), m.group(2))); continue
        m = _re.search(r'the (\w+) block is on the table', fl)
        if m: s['on_table'].append(m.group(1)); continue
        m = _re.search(r'the (\w+) block is clear', fl)
        if m: s['clear'].append(m.group(1)); continue
        m = _re.search(r'holding (?:the )?(\w+) block', fl)
        if m: s['holding'] = m.group(1); continue
        if 'hand is empty' in fl: s['hand_empty'] = True
    return s

def _bw_apply(state, action):
    s = copy.deepcopy(state); a = action.strip().lower()
    m = _re.match(r'unstack (?:the )?(\w+) block from (?:on top of )?(?:the )?(\w+) block', a)
    if m:
        X, Y = m.group(1), m.group(2)
        s['on'] = [(a2,b2) for a2,b2 in s['on'] if not (a2==X and b2==Y)]
        s['clear'] = [c for c in s['clear'] if c != X]
        s['hand_empty'] = False; s['holding'] = X
        if Y not in s['clear']: s['clear'].append(Y)
        return s
    m = _re.match(r'pick up (?:the )?(\w+) block', a)
    if m:
        X = m.group(1); s['on_table'] = [b for b in s['on_table'] if b != X]
        s['clear'] = [c for c in s['clear'] if c != X]
        s['hand_empty'] = False; s['holding'] = X; return s
    m = _re.match(r'put down (?:the )?(\w+) block', a)
    if m:
        X = m.group(1); s['holding'] = None; s['hand_empty'] = True
        if X not in s['on_table']: s['on_table'].append(X)
        if X not in s['clear']:    s['clear'].append(X)
        return s
    m = _re.match(r'stack (?:the )?(\w+) block on (?:top of )?(?:the )?(\w+) block', a)
    if m:
        X, Y = m.group(1), m.group(2); s['holding'] = None; s['hand_empty'] = True
        s['clear'] = [c for c in s['clear'] if c != Y]
        if X not in s['clear']: s['clear'].append(X)
        if (X, Y) not in s['on']: s['on'].append((X, Y))
        return s
    return None

# -- Mystery state helpers (rel1..rel5) ------------------------------------
# State is a plain set of fact strings e.g. {'rel1 object a', 'rel3'}
# _mystery_apply calls _abstract_mystery internally so raw action names work.
def _mystery_apply(state, action):
    a = _abstract_mystery(action.strip().lower()); s = set(state)
    m = _re.match(r'action1 object (\w+)$', a)
    if m:
        x = m.group(1)
        s -= {f'rel1 object {x}', f'rel2 object {x}', 'rel3'}
        s.add(f'rel4 object {x}'); return s
    m = _re.match(r'action2 object (\w+)$', a)
    if m:
        x = m.group(1); s.discard(f'rel4 object {x}')
        s |= {f'rel1 object {x}', f'rel2 object {x}', 'rel3'}; return s
    m = _re.match(r'action3 object (\w+) from object (\w+)$', a)
    if m:
        x, y = m.group(1), m.group(2)
        s -= {f'rel1 object {y}', f'rel4 object {x}'}
        s |= {'rel3', f'rel1 object {x}', f'object {x} rel5 object {y}'}; return s
    m = _re.match(r'action4 object (\w+) from object (\w+)$', a)
    if m:
        x, y = m.group(1), m.group(2)
        s -= {f'object {x} rel5 object {y}', f'rel1 object {x}', 'rel3'}
        s |= {f'rel4 object {x}', f'rel1 object {y}'}; return s
    return None

# -- Logistics state helpers --------------------------------------------------
def _logistics_parse(facts):
    locs = {}; airports = set(); city_map = {}
    for f in facts:
        fl = f.lower().rstrip('.')
        if 'is an airport' in fl:
            m = _re.search(r'(\S+) is an airport', fl)
            if m: airports.add(m.group(1))
        elif 'is in city' in fl:
            m = _re.search(r'(\S+) is in city (\S+)', fl)
            if m: city_map[m.group(1)] = m.group(2)
        elif 'is at' in fl:
            m = _re.search(r'(\S+) is at (\S+)', fl)
            if m: locs[m.group(1)] = m.group(2)
        elif 'is in' in fl:
            m = _re.search(r'(\S+) is in (\S+)', fl)
            if m: locs[m.group(1)] = m.group(2)
    return {'locs': locs, 'airports': airports, 'city_map': city_map}

def _logistics_apply(state, action):
    s = copy.deepcopy(state); a = action.strip().lower()
    m = _re.match(r'load (\S+) into (\S+) at (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(2); return s
    m = _re.match(r'unload (\S+) from (\S+) at (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(3); return s
    m = _re.match(r'drive (\S+) from (\S+) to (\S+) in (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(3); return s
    m = _re.match(r'fly (\S+) from (\S+) to (\S+)', a)
    if m: s['locs'][m.group(1)] = m.group(3); return s
    return None

# -- Convert state to frozenset of fact strings ------------------------------
def _to_fact_set(state, domain):
    if domain in ('blocksworld', 'blocksworld_3'):
        facts = []
        for x, y in state['on']:     facts.append(f'the {x} block is on top of the {y} block')
        for x in state['on_table']:  facts.append(f'the {x} block is on the table')
        for x in state['clear']:     facts.append(f'the {x} block is clear')
        if state['holding']:         facts.append(f'you are holding the {state["holding"]} block')
        if state['hand_empty']:      facts.append('the hand is empty')
        return frozenset(facts)
    elif domain in ('mystery', 'mystery_3'):
        return frozenset(state)   # state is already a set of fact strings
    elif domain == 'logistics':
        facts = []
        for e, loc in state['locs'].items():
            facts.append(f'{e} is in {loc}' if any(loc.startswith(v)
                         for v in ('truck', 'airplane')) else f'{e} is at {loc}')
        return frozenset(facts)
    return frozenset()

# -- Goal check after all steps pass ----------------------------------------
def goal_check(plan_recs, domain, goal_facts):
    '''
    Verify final world state matches goal after all actions in plan.
    Called when stepwise evaluator has predicted A for every step.
    Rebuilds state from scratch using first record state_facts (initial state),
    then applies each action sequentially.
    Returns 1 (valid) or 0 (invalid - goal not reached).
    '''
    recs = sorted(plan_recs, key=lambda x: x['step_idx'])
    if domain in ('blocksworld', 'blocksworld_3'):
        state    = _bw_parse(recs[0]['state_facts'])
        apply_fn = _bw_apply
    elif domain in ('mystery', 'mystery_3'):
        # state_facts use rel1..rel5 (written by notebook 00)
        state    = {f.lower().rstrip('.') for f in recs[0]['state_facts']}
        apply_fn = _mystery_apply
    elif domain == 'logistics':
        state    = _logistics_parse(recs[0]['state_facts'])
        apply_fn = _logistics_apply
    else:
        return 0
    for rec in recs:
        # rec['action'] stores raw action string e.g. 'attack object a'
        # _mystery_apply calls _abstract_mystery internally, so this is safe.
        ns = apply_fn(state, rec['action'])
        if ns is not None: state = ns
    final_facts = _to_fact_set(state, domain)
    return 1 if all(g.lower().rstrip('.') in final_facts for g in goal_facts) else 0

print('State helpers & goal check: OK')
print('Mystery abstraction: rel1..rel5 (matches 00_data_preparation.ipynb)')

## 3. Model Loader and Batched Inference

`load_model(adapter_path, tok_path)` loads Flan-T5-XL with an optional LoRA
adapter. When `adapter_path=None`, the base model is used without any adapter
(zero-shot conditions: `FlanT5_Direct` and `FlanT5_Stepwise`).

`batch_predict(model, tokenizer, prompts, max_len)` runs batched generation
with greedy decoding (`num_beams=1, do_sample=False`) and returns the first
output token stripped to a single character (`A` or `B`).

In [ ]:
def load_model(adapter_path=None, tok_path=None):
    '''Load Flan-T5-XL with optional LoRA adapter for evaluation.'''
    cleanup()
    tok  = AutoTokenizer.from_pretrained(tok_path or BASE_MODEL, use_fast=True)
    base = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16 if use_bf16 else torch.float32,
        device_map='auto',
    )
    if adapter_path and os.path.isdir(adapter_path):
        model = PeftModel.from_pretrained(base, adapter_path)
        print(f'  Adapter loaded: {adapter_path}')
    else:
        model = base
        print('  Base model (no adapter)')
    model.eval()
    return model, tok

def normalize_pred(text):
    '''Extract A or B from model output. Default B on empty/unrecognisable.'''
    if not text: return 'B'
    c = text.strip()[0].upper()
    return c if c in {'A', 'B'} else 'B'

@torch.no_grad()
def batch_predict(model, tokenizer, prompts, batch_size=BATCH_SIZE,
                  max_len=MAX_SOURCE_LEN_SW):
    '''
    Run batched inference. Returns list of 'A' or 'B' predictions.
    Uses model.generate() for consistent decoding.
    '''
    predictions = []
    for i in range(0, len(prompts), batch_size):
        batch  = prompts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            max_length=max_len, truncation=True
        ).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=MAX_TARGET_LEN)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([normalize_pred(d) for d in decoded])
    return predictions

print('Model loader & inference: OK')

## 4. Load Test Datasets

Loads all test records written by `00_data_preparation.ipynb`:

- `planbench_stepwise_test.jsonl` — one record per action step, for stepwise evaluation
- `planbench_direct_test.jsonl` — one record per plan, for direct evaluation
- `direct_splits.json` — plan_id splits for FT_Random holdout evaluation

`FT_Random` is evaluated on its own 20% holdout (plans whose plan_ids are in
`splits['random']['test_plan_ids']`), not the full test set, because 70% of
the test set was used for its training.

In [ ]:
# Load stepwise test records
sw_all = [json.loads(l) for l in open(SW_TEST_PATH)]
sw_by_domain = defaultdict(list)
for r in sw_all: sw_by_domain[r['domain']].append(r)
print(f'Stepwise test records: {len(sw_all)}')
for d in DOMAINS:
    cnt = Counter(r['label'] for r in sw_by_domain[d])
    print(f'  {d:15s}: {len(sw_by_domain[d]):5d} steps  (A={cnt["A"]} B={cnt["B"]})')

# Load direct test records
di_all = [json.loads(l) for l in open(DI_TEST_PATH)]
di_by_domain = defaultdict(list)
for r in di_all: di_by_domain[r['domain']].append(r)
print(f'\nDirect test records: {len(di_all)}')
for d in DOMAINS:
    cnt = Counter(r['label'] for r in di_by_domain[d])
    print(f'  {d:15s}: {len(di_by_domain[d]):5d} plans  (A={cnt["A"]} B={cnt["B"]})')

# Load FT_Random / FT_Length test plan_ids from direct_splits.json
splits = json.load(open(SPLITS_PATH))
rand_test_ids = {d: set(splits['random']['test_plan_ids'][d]) for d in DOMAINS}
len_test_ids  = {d: set(splits['length']['test_plan_ids'][d]) for d in DOMAINS}
print(f'\nFT_Random test plans per domain: '
      f'{[len(rand_test_ids[d]) for d in DOMAINS]}')
print(f'FT_Length test plans per domain: '
      f'{[len(len_test_ids[d]) for d in DOMAINS]}')

# Build filtered direct records for FT_Random and FT_Length
rand_di_by_domain = {
    d: [r for r in di_by_domain[d] if r['plan_id'] in rand_test_ids[d]]
    for d in DOMAINS
}
len_di_by_domain = {
    d: [r for r in di_by_domain[d] if r['plan_id'] in len_test_ids[d]]
    for d in DOMAINS
}

# Debug mode: truncate to DEBUG_N plans per domain
if DEBUG:
    print(f'\n[DEBUG] Truncating to {DEBUG_N} plans/domain')
    for d in DOMAINS:
        # Stepwise: get plan_ids of first DEBUG_N plans
        sw_pids = sorted(set(r['plan_id'] for r in sw_by_domain[d]))[:DEBUG_N]
        sw_by_domain[d] = [r for r in sw_by_domain[d] if r['plan_id'] in sw_pids]
        # Direct: first DEBUG_N plans
        di_by_domain[d]   = di_by_domain[d][:DEBUG_N]
        rand_di_by_domain[d] = rand_di_by_domain[d][:DEBUG_N]
        len_di_by_domain[d]  = len_di_by_domain[d][:DEBUG_N]
    print(f'  SW records: {sum(len(sw_by_domain[d]) for d in DOMAINS)}')
    print(f'  DI records: {sum(len(di_by_domain[d]) for d in DOMAINS)}')

## 5. Smoke Test

Runs a quick sanity check before the full evaluation:
1. Verifies all data files and adapter directories exist
2. Loads the base model and runs inference on 2 sample prompts
3. Confirms the output format is correct (single token `A` or `B`)

Run this cell before starting the full evaluation to catch any setup issues early.

In [ ]:
def run_smoke_test():
    # 1. Data files exist
    assert os.path.exists(SW_TEST_PATH), f'MISSING: {SW_TEST_PATH}'
    assert os.path.exists(DI_TEST_PATH), f'MISSING: {DI_TEST_PATH}'
    assert os.path.exists(SPLITS_PATH),  f'MISSING: {SPLITS_PATH}'
    # 2. All domains have records
    for d in DOMAINS:
        assert len(sw_by_domain[d]) > 0, f'No SW records for {d}'
        assert len(di_by_domain[d]) > 0, f'No DI records for {d}'
    # 3. Required fields present
    sw_rec = sw_by_domain[DOMAINS[0]][0]
    for f in ['plan_id','step_idx','step_num','label','label_int',
              'action','state_facts','rules','goal_facts','plan_label']:
        assert f in sw_rec, f'SW record missing field: {f}'
    di_rec = di_by_domain[DOMAINS[0]][0]
    for f in ['plan_id','label_int','prompt','n_actions']:
        assert f in di_rec, f'DI record missing field: {f}'
    # 4. Mystery state_facts use rel1..rel5
    mys_recs = sw_by_domain.get('mystery', [])
    if mys_recs:
        sample_facts = ' '.join(mys_recs[0]['state_facts'])
        assert 'rel1' not in sample_facts and 'rel2' not in sample_facts, \
            f'Mystery state_facts still use old rel naming: {sample_facts[:100]}'
        print('  Mystery naming: rel1..rel5 confirmed')
    # 5. Adapter paths exist
    adapters = [
        ('FlanT5_Stepwise_FT_Combined', ADAPTER_COMBINED),
        ('FlanT5_Stepwise_FT_FOLIO',    ADAPTER_FOLIO),
        ('FlanT5_Stepwise_FT_PB',       ADAPTER_PB),
        ('FlanT5_Direct_FT50',           ADAPTER_FT50),
        ('FlanT5_Direct_FT_Random',      ADAPTER_RANDOM),
        ('FlanT5_Direct_FT_Length',      ADAPTER_LENGTH),
    ]
    for name, path in adapters:
        status = 'OK' if os.path.isdir(path) else 'MISSING'
        print(f'  {name:35s}: {status}')
    # 6. Quick inference test with base model
    print('  Testing inference pipeline...')
    model, tok = load_model()  # base model, no adapter
    sample = sw_by_domain[DOMAINS[0]][0]['prompt']
    preds  = batch_predict(model, tok, [sample], batch_size=1)
    assert preds[0] in ('A', 'B'), f'Bad prediction: {preds[0]}'
    print(f'  Inference OK: prediction={preds[0]}')
    del model; cleanup()
    print('\nSmoke test PASSED')

run_smoke_test()

## 6. Direct Condition Evaluator

`evaluate_direct(model, tokenizer, di_records, domain, cond_name)` evaluates
one direct condition on one domain.

Each plan record contains the full plan prompt. The model predicts `A` (valid)
or `B` (invalid) in a single forward pass. No state tracking or goal checking
is performed — the model must assess the entire plan at once.

Returns a result dict with plan-level metrics (plan_f1, plan_acc, confusion matrix)
and raw prediction arrays (`plan_golds`, `plan_preds`).

In [ ]:
def evaluate_direct(model, tokenizer, di_records, domain, cond_name):
    '''
    Evaluate one direct condition on one domain.
    Returns a result dict with plan-level metrics.
    '''
    if not di_records:
        print(f'  [{cond_name}|{domain}] no records - skipping')
        return None
    prompts = [r['prompt']    for r in di_records]
    golds   = [r['label_int'] for r in di_records]
    preds_raw = batch_predict(model, tokenizer, prompts,
                             max_len=MAX_SOURCE_LEN_DI)
    preds     = [1 if p == 'A' else 0 for p in preds_raw]
    acc = accuracy_score(golds, preds)
    f1  = f1_score(golds, preds, average='macro', zero_division=0)
    cm  = confusion_matrix(golds, preds, labels=[0, 1])
    print(f'  [{cond_name}|{domain}] n={len(golds):4d}  '
          f'plan_acc={acc:.4f}  plan_f1={f1:.4f}')
    return {
        'cond'       : cond_name,
        'domain'     : domain,
        'eval_type'  : 'direct',
        'n_plans'    : len(golds),
        'plan_acc'   : round(acc, 4),
        'plan_f1'    : round(f1,  4),
        'plan_cm'    : cm.tolist(),
        'plan_golds' : golds,
        'plan_preds' : preds,
    }

## 7. Stepwise Condition Evaluator

`evaluate_stepwise(model, tokenizer, sw_records, domain, cond_name)` evaluates
one stepwise condition on one domain.

**Pipeline logic:**
1. All step prompts for all plans are batched together for efficient GPU inference
2. Per-plan: steps are processed in order (by step_idx)
3. First predicted `B` → plan marked invalid, failing step recorded, loop breaks
4. All steps predicted `A` → `goal_check()` called to determine final verdict

**Action-level evaluation:** collected for every step processed before the
pipeline stops. Steps never reached (after a predicted B) are excluded.

**Failing step accuracy:** computed only for invalid plans where the model
also predicted invalid (predicted B at some step). Measures whether the model
stopped at the correct failing step.

In [ ]:
def evaluate_stepwise(model, tokenizer, sw_records, domain, cond_name):
    '''
    Evaluate one stepwise condition on one domain.
    Batches all step prompts for efficiency, then applies plan-level logic.
    Returns result dict with plan-level and action-level metrics.
    '''
    # Group records by plan_id, sort by step_idx
    plans = defaultdict(list)
    for r in sw_records: plans[r['plan_id']].append(r)
    for pid in plans: plans[pid].sort(key=lambda x: x['step_idx'])
    plan_ids = sorted(plans.keys())
    if not plan_ids:
        print(f'  [{cond_name}|{domain}] no records - skipping')
        return None

    # Flatten all step prompts for batched prediction
    flat_prompts = []
    flat_meta    = []   # (plan_id, step_idx, step_num, label_int, goal_facts, plan_label)
    for pid in plan_ids:
        for r in plans[pid]:
            flat_prompts.append(r['prompt'])
            flat_meta.append((
                pid, r['step_idx'], r['step_num'],
                r['label_int'], r['goal_facts'], r['plan_label'],
            ))

    print(f'  [{cond_name}|{domain}] {len(plan_ids)} plans, '
          f'{len(flat_prompts)} steps ...')
    flat_preds = batch_predict(model, tokenizer, flat_prompts,
                             max_len=MAX_SOURCE_LEN_SW)

    # Reconstruct per-plan step results
    plan_step_preds = defaultdict(list)  # pid -> [(step_idx, step_num, pred, gold)]
    for (pid, sidx, snum, sgold, _, _), pred in zip(flat_meta, flat_preds):
        plan_step_preds[pid].append((sidx, snum, pred, sgold))

    # Plan-level and action-level evaluation
    plan_golds, plan_preds_out  = [], []
    action_golds, action_preds  = [], []
    fail_step_golds, fail_step_preds = [], []
    err_action, err_goal        = 0, 0

    for pid in plan_ids:
        steps     = sorted(plan_step_preds[pid], key=lambda x: x[0])
        plan_gold = plans[pid][0]['plan_label']
        goal_facts= plans[pid][0]['goal_facts']

        plan_pred    = 1
        stopped      = False
        step_mistake = False
        pred_fail_step = None

        for sidx, snum, pred, sgold in steps:
            action_golds.append(sgold)
            action_preds.append(1 if pred == 'A' else 0)
            if sgold != (1 if pred == 'A' else 0):
                step_mistake = True
            if pred == 'B':
                plan_pred      = 0
                pred_fail_step = snum
                stopped        = True
                break

        if not stopped:
            plan_pred = goal_check(plans[pid], domain, goal_facts)

        plan_golds.append(plan_gold)
        plan_preds_out.append(plan_pred)

        # Failing step accuracy: only for invalid plans predicted as invalid
        if plan_gold == 0:
            # Find gold failing step from records
            gold_fail = next(
                (r['step_num'] for r in plans[pid] if r['label_int'] == 0),
                None
            )
            if gold_fail is not None and pred_fail_step is not None:
                fail_step_golds.append(gold_fail)
                fail_step_preds.append(pred_fail_step)

        if plan_gold != plan_pred:
            if step_mistake: err_action += 1
            else:            err_goal   += 1

    # Compute metrics
    acc_plan = accuracy_score(plan_golds, plan_preds_out)
    f1_plan  = f1_score(plan_golds, plan_preds_out, average='macro', zero_division=0)
    cm_plan  = confusion_matrix(plan_golds, plan_preds_out, labels=[0, 1])
    acc_act  = accuracy_score(action_golds, action_preds) if action_golds else None
    f1_act   = f1_score(action_golds, action_preds, average='macro',
                        zero_division=0) if action_golds else None
    cm_act   = confusion_matrix(action_golds, action_preds,
                                labels=[0, 1]) if action_golds else None
    # Failing step accuracy
    fs_acc = (sum(g == p for g, p in zip(fail_step_golds, fail_step_preds))
              / max(1, len(fail_step_golds))) if fail_step_golds else None

    print(f'  [{cond_name}|{domain}] '
          f'plan_acc={acc_plan:.4f}  plan_f1={f1_plan:.4f}  '
          f'act_acc={acc_act:.4f}  '
          f'fail_step_acc={fs_acc:.4f}' if acc_act and fs_acc else
          f'  [{cond_name}|{domain}] '
          f'plan_acc={acc_plan:.4f}  plan_f1={f1_plan:.4f}')

    return {
        'cond'             : cond_name,
        'domain'           : domain,
        'eval_type'        : 'stepwise',
        'n_plans'          : len(plan_golds),
        'n_actions'        : len(action_golds),
        'plan_acc'         : round(acc_plan, 4),
        'plan_f1'          : round(f1_plan,  4),
        'plan_cm'          : cm_plan.tolist(),
        'action_acc'       : round(acc_act, 4) if acc_act is not None else None,
        'action_f1'        : round(f1_act,  4) if f1_act  is not None else None,
        'action_cm'        : cm_act.tolist()   if cm_act  is not None else None,
        'failing_step_acc' : round(fs_acc, 4)  if fs_acc  is not None else None,
        'n_failing_step_evaluated': len(fail_step_golds),
        'n_errors'         : sum(g != p for g, p in zip(plan_golds, plan_preds_out)),
        'err_action'       : err_action,
        'err_goal'         : err_goal,
        'plan_golds'       : plan_golds,
        'plan_preds'       : plan_preds_out,
        'action_golds'     : action_golds,
        'action_preds'     : action_preds,
        'fail_step_golds'  : fail_step_golds,
        'fail_step_preds'  : fail_step_preds,
    }

## 8. Run All Conditions

Runs all FlanT5 conditions sequentially. Each condition:
1. Loads the model (base + optional LoRA adapter)
2. Runs evaluation on all 5 domains
3. Frees GPU memory before loading the next model

Estimated runtime: ~20-30 minutes per condition on an A100 GPU.
Total: ~3-4 hours for all 7 conditions.

In [ ]:
all_results = {}
SEP = '=' * 60

# =============================================================================
# DIRECT CONDITIONS
# =============================================================================

DIRECT_CONDS = [
    # (cond_name, adapter_path, tok_path, di_records_dict)
    ('FlanT5_Direct',
     None,          None,       di_by_domain),
    ('FlanT5_Direct_FT50',
     ADAPTER_FT50,  TOK_FT50,   di_by_domain),
    ('FlanT5_Direct_FT_Random',
     ADAPTER_RANDOM, TOK_RANDOM, rand_di_by_domain),
]

for cond_name, adapter, tok_path, records_dict in DIRECT_CONDS:
    print(f'\n{SEP}')
    print(f'CONDITION: {cond_name}  [direct]')
    print(SEP)
    model, tok = load_model(adapter, tok_path)
    all_results[cond_name] = {}
    for domain in DOMAINS:
        r = evaluate_direct(model, tok, records_dict[domain], domain, cond_name)
        if r: all_results[cond_name][domain] = r
    del model; cleanup()
    mean_f1 = np.mean([all_results[cond_name][d]['plan_f1']
                       for d in DOMAINS if d in all_results[cond_name]])
    print(f'  Mean plan_f1 = {mean_f1:.4f}')

# =============================================================================
# STEPWISE CONDITIONS
# =============================================================================

STEPWISE_CONDS = [
    # (cond_name, adapter_path, tok_path)
    ('FlanT5_Stepwise',
     None,            None),
    ('FlanT5_Stepwise_FT_FOLIO',
     ADAPTER_FOLIO,   TOK_FOLIO),
    ('FlanT5_Stepwise_FT_PB',
     ADAPTER_PB,      TOK_PB),
    ('FlanT5_Stepwise_FT_Combined',
     ADAPTER_COMBINED, TOK_COMBINED),
]

for cond_name, adapter, tok_path in STEPWISE_CONDS:
    print(f'\n{SEP}')
    print(f'CONDITION: {cond_name}  [stepwise]')
    print(SEP)
    model, tok = load_model(adapter, tok_path)
    all_results[cond_name] = {}
    for domain in DOMAINS:
        r = evaluate_stepwise(model, tok, sw_by_domain[domain], domain, cond_name)
        if r: all_results[cond_name][domain] = r
    del model; cleanup()
    mean_f1 = np.mean([all_results[cond_name][d]['plan_f1']
                       for d in DOMAINS if d in all_results[cond_name]])
    print(f'  Mean plan_f1 = {mean_f1:.4f}')

print(f'\n{SEP}')
print('ALL CONDITIONS COMPLETE')

## 9. Summary Tables

Prints results grouped by metric:
- Plan F1-macro across all conditions and domains
- Action F1-macro for stepwise conditions only
- Failing step accuracy for stepwise conditions only
- Plan accuracy and action accuracy

`FT_Random` results are on its own 20% holdout and are marked accordingly.

In [ ]:
SEP2 = '-' * 85
DOM_SHORT = ['bw', 'bw3', 'mys', 'mys3', 'log']

def print_table(title, cond_names, metric):
    print(f'\n{SEP2}')
    print(title)
    print(f'{"Condition":<35s}' + ''.join(f'{s:>8s}' for s in DOM_SHORT) + f'{"Mean":>8s}')
    print(SEP2)
    for cond in cond_names:
        if cond not in all_results: continue
        vals = [all_results[cond].get(d, {}).get(metric) for d in DOMAINS]
        valid = [v for v in vals if v is not None]
        mean  = np.mean(valid) if valid else float('nan')
        row   = f'{cond:<35s}' + ''.join(
            f'{v:>8.4f}' if v is not None else f'{"N/A":>8s}' for v in vals
        )
        print(row + f'{mean:>8.4f}')

DIRECT_COND_NAMES   = ['FlanT5_Direct', 'FlanT5_Direct_FT50',
                       'FlanT5_Direct_FT_Random', 'FlanT5_Direct_FT_Length']
STEPWISE_COND_NAMES = ['FlanT5_Stepwise', 'FlanT5_Stepwise_FT_FOLIO',
                       'FlanT5_Stepwise_FT_PB', 'FlanT5_Stepwise_FT_Combined']

print_table('DIRECT — Plan F1-macro',    DIRECT_COND_NAMES,   'plan_f1')
print_table('DIRECT — Plan Accuracy',    DIRECT_COND_NAMES,   'plan_acc')
print()
print_table('STEPWISE — Plan F1-macro',  STEPWISE_COND_NAMES, 'plan_f1')
print_table('STEPWISE — Plan Accuracy',  STEPWISE_COND_NAMES, 'plan_acc')
print_table('STEPWISE — Action Accuracy',STEPWISE_COND_NAMES, 'action_acc')
print_table('STEPWISE — Action F1-macro',STEPWISE_COND_NAMES, 'action_f1')
print_table('STEPWISE — Failing Step Accuracy', STEPWISE_COND_NAMES, 'failing_step_acc')

print()
print('NOTE: FlanT5_Direct_FT_Random and FT_Length are evaluated on their')
print('own 20% held-out subsets, not the full test set.')
print('Their Plan F1 values are NOT directly comparable to other conditions.')

## 10. Save Results

Saves two files to `paper2/results/`:

**`results_flant5.json`** — all computed metrics (F1, accuracy, confusion matrices)
for every condition and domain. Raw prediction arrays are excluded to keep the
file small.

**`raw_preds_flant5.json`** — raw prediction arrays (`plan_golds`, `plan_preds`,
`action_golds`, `action_preds`, `fail_step_golds`, `fail_step_preds`) for every
condition and domain. Used by `06_additional_metrics.ipynb` to compute alternative
metrics (e.g., weighted F1) without re-running inference.

In [ ]:
SKIP_KEYS = {
    'plan_golds', 'plan_preds',
    'action_golds', 'action_preds',
    'fail_step_golds', 'fail_step_preds',
}

def serialise(r):
    return {k: v for k, v in r.items() if k not in SKIP_KEYS}

save_dict = {
    cond: {domain: serialise(r) for domain, r in domain_results.items()}
    for cond, domain_results in all_results.items()
}

out_path = os.path.join(RESULTS_DIR, 'results_flant5.json')
with open(out_path, 'w') as f:
    json.dump(save_dict, f, indent=2)
print(f'Results saved -> {out_path}')

# Quick summary
print()
print('FINAL PLAN F1-MACRO SUMMARY')
print(SEP2)
all_conds = DIRECT_COND_NAMES + STEPWISE_COND_NAMES
for cond in all_conds:
    if cond not in all_results: continue
    vals  = [all_results[cond].get(d, {}).get('plan_f1') for d in DOMAINS]
    valid = [v for v in vals if v is not None]
    mean  = np.mean(valid) if valid else float('nan')
    etype = all_results[cond].get(DOMAINS[0], {}).get('eval_type', '?')
    print(f'  {cond:<35s} [{etype:8s}]  mean_f1={mean:.4f}')
print()
print('Next step: run 03_eval_gpt4.ipynb')

In [ ]:
# Save raw prediction arrays for additional metrics computation
# (e.g. weighted F1) without re-running inference in a separate notebook.
RAW_KEYS = {
    'plan_golds', 'plan_preds',
    'action_golds', 'action_preds',
    'fail_step_golds', 'fail_step_preds',
}

raw_dict = {}
for cond, domain_results in all_results.items():
    raw_dict[cond] = {}
    for domain, r in domain_results.items():
        raw_dict[cond][domain] = {k: v for k, v in r.items() if k in RAW_KEYS}

raw_path = os.path.join(RESULTS_DIR, 'raw_preds_flant5.json')
with open(raw_path, 'w') as f:
    json.dump(raw_dict, f)
print(f'Raw predictions saved -> {raw_path}')
print(f'Conditions: {list(raw_dict.keys())}')
print(f'Keys per result: {list(RAW_KEYS)}')

## 11. Confusion Matrices

Per-condition per-domain confusion matrices for plan-level binary classification.

- Rows: gold label (0=Invalid, 1=Valid)
- Cols: predicted label (0=Invalid, 1=Valid)
- TN: invalid correctly predicted invalid
- FP: invalid wrongly predicted valid
- FN: valid wrongly predicted invalid
- TP: valid correctly predicted valid

In [ ]:
def print_confusion_matrices():
    SEP3 = '=' * 70
    all_conds = DIRECT_COND_NAMES + STEPWISE_COND_NAMES
    for cond in all_conds:
        if cond not in all_results: continue
        print(f'\n{SEP3}')
        print(f'CONFUSION MATRICES — {cond}')
        print(f'{SEP3}')
        print(f'  {"Domain":<15s}  {"TN":>6s}  {"FP":>6s}  {"FN":>6s}  {"TP":>6s}  '
              f'{"Precision":>10s}  {"Recall":>8s}')
        print('  ' + '-' * 65)
        for domain in DOMAINS:
            r = all_results[cond].get(domain)
            if not r: continue
            cm = r['plan_cm']  # [[TN,FP],[FN,TP]] labels=[0,1]
            tn, fp = cm[0][0], cm[0][1]
            fn, tp = cm[1][0], cm[1][1]
            prec = tp / max(1, tp+fp)
            rec  = tp / max(1, tp+fn)
            print(f'  {domain:<15s}  {tn:>6d}  {fp:>6d}  {fn:>6d}  {tp:>6d}  '
                  f'{prec:>10.4f}  {rec:>8.4f}')

print_confusion_matrices()


## 12. Grouped Results: Abstract vs Non-Abstract Domains

Groups results by domain type to highlight the Mystery domain challenge:

- **Non-abstract domains** (Blocksworld, Blocksworld-3, Logistics): use natural
  language vocabulary — surface-level patterns may help
- **Abstract domains** (Mystery, Mystery-3): use fully abstract vocabulary
  (Rel1..Rel5, Action1..Action4) — model must reason from preconditions only

This grouping reveals whether a condition genuinely learned logical reasoning
(transfers to abstract domains) or merely learned surface patterns
(fails on abstract domains).

In [ ]:
NON_ABSTRACT = ['blocksworld', 'blocksworld_3', 'logistics']
ABSTRACT     = ['mystery', 'mystery_3']

METRICS = [
    ('plan_f1',          'Plan F1-macro'),
    ('plan_acc',         'Plan Accuracy'),
    ('action_f1',        'Action F1-macro'),
    ('action_acc',       'Action Accuracy'),
    ('failing_step_acc', 'Failing Step Accuracy'),
]

def group_mean(cond, domains, metric):
    vals = [all_results[cond].get(d, {}).get(metric)
            for d in domains if cond in all_results]
    vals = [v for v in vals if v is not None]
    return np.mean(vals) if vals else None

def print_grouped(cond_names, group_label):
    print(f'\n{"-"*80}')
    print(f'{group_label}')
    print(f'{"-"*80}')
    header = f'{"Condition":<35s}  {"Non-Abstract":>13s}  {"Abstract":>10s}'
    for metric, label in METRICS:
        # Skip action/failing_step for direct conditions
        is_stepwise_metric = metric in ('action_f1','action_acc','failing_step_acc')
        has_any = any(
            all_results.get(c, {}).get(d, {}).get(metric) is not None
            for c in cond_names for d in DOMAINS
        )
        if not has_any: continue
        print(f'\n  {label}')
        print(f'  {"Condition":<35s}  {"Non-Abstract":>13s}  {"Abstract":>10s}')
        print('  ' + '-' * 63)
        for cond in cond_names:
            if cond not in all_results: continue
            na = group_mean(cond, NON_ABSTRACT, metric)
            ab = group_mean(cond, ABSTRACT, metric)
            na_str = f'{na:.4f}' if na is not None else '   N/A'
            ab_str = f'{ab:.4f}' if ab is not None else '   N/A'
            print(f'  {cond:<35s}  {na_str:>13s}  {ab_str:>10s}')

print('=' * 80)
print('GROUPED RESULTS: NON-ABSTRACT vs ABSTRACT DOMAINS')
print('Non-abstract: blocksworld, blocksworld_3, logistics')
print('Abstract    : mystery, mystery_3')
print_grouped(DIRECT_COND_NAMES,   'DIRECT CONDITIONS')
print_grouped(STEPWISE_COND_NAMES, 'STEPWISE CONDITIONS')
